In [ ]:

import re
import random
from collections import defaultdict, Counter

# ==========================================
# MARKOV CHAIN TEXT GENERATOR
# ==========================================

class MarkovTextGenerator:
    def __init__(self, order=2):
        self.order = order
        self.chain = defaultdict(Counter)
        self.vocabulary = set()

    def preprocess(self, text):
        """Normalize text and tokenize it into words and punctuation."""
        text = text.lower()
        return re.findall(r"\w+|[.,!?;]", text)

    def train(self, text):
        """Learn transition probabilities from the training corpus."""
        tokens = self.preprocess(text)

        if len(tokens) <= self.order:
            raise ValueError("Training text is too short for the selected order.")

        self.vocabulary = set(tokens)

        for i in range(len(tokens) - self.order):
            state = tuple(tokens[i:i + self.order])
            next_token = tokens[i + self.order]
            self.chain[state][next_token] += 1

        print(f"Training complete.")
        print(f"Model order: {self.order}")
        print(f"States learned: {len(self.chain)}")
        print(f"Vocabulary size: {len(self.vocabulary)}")

    def generate(self, length=50, seed=None):
        """Generate text by sampling learned transition probabilities."""
        if not self.chain:
            raise ValueError("Train the model before generating text.")

        if length <= 0:
            raise ValueError("Length must be positive.")

        if seed:
            seed_tokens = self.preprocess(seed)
            if len(seed_tokens) < self.order:
                raise ValueError(
                    f"Seed must contain at least {self.order} tokens."
                )
            state = tuple(seed_tokens[-self.order:])
            if state not in self.chain:
                raise ValueError("Seed context not found in training data.")
            output = seed_tokens[:]
        else:
            state = random.choice(list(self.chain.keys()))
            output = list(state)

        for _ in range(length):
            transitions = self.chain.get(state)
            if not transitions:
                break

            candidates = list(transitions.keys())
            weights = list(transitions.values())
            next_token = random.choices(candidates, weights=weights, k=1)[0]

            output.append(next_token)
            state = tuple(output[-self.order:])

        # Restore readable spacing around punctuation
        result = " ".join(output)
        result = re.sub(r"\s+([.,!?;])", r"\1", result)
        return result.capitalize()


# ==========================================
# TRAINING CORPUS
# ==========================================

training_text = """
Artificial intelligence is transforming modern technology.
Machine learning enables computers to learn patterns from data.
Natural language processing helps computers understand human language.
Text generation is an important application of natural language processing.
Markov chains model transitions between words using probabilities.
A Markov chain predicts the next word based on previous words.
Machine learning models require data for training and evaluation.
Artificial intelligence is used in education, healthcare, and research.
Natural language processing supports chatbots and text generation systems.
Data science combines statistics, programming, and machine learning.
"""

# ==========================================
# TRAIN AND GENERATE
# ==========================================

model = MarkovTextGenerator(order=2)
model.train(training_text)

print("\n--- GENERATED TEXT ---")
generated_text = model.generate(
    length=40,
    seed="machine learning"
)
print(generated_text)


Training complete.
Model order: 2
States learned: 87
Vocabulary size: 64

--- GENERATED TEXT ---
Machine learning enables computers to learn patterns from data. natural language processing helps computers understand human language. text generation is an important application of natural language processing helps computers understand human language. text generation is an important application of
